In [1]:
from ipyleaflet import Map, basemaps, TileLayer, GeoJSON, WidgetControl
from ipywidgets import HTML
import json
import urllib.request
import subsample_pyramid
import xarray as xr
import geopandas as gpd


In [4]:
center = [-7.5, -65.5]
zoom = 5

m = Map(basemap=basemaps.Esri.WorldImagery, 
    center=center, 
    zoom=zoom)

In [5]:
hydrobasins_lev05_url = 'https://raw.githubusercontent.com/blackteacatsu/spring_2024_envs_research_amazon_ldas/main/resources/hybas_sa_lev05_areaofstudy.geojson'

with urllib.request.urlopen(hydrobasins_lev05_url) as url:
    geojson_data = json.loads(url.read().decode())

polygon_layer = GeoJSON(
    data=geojson_data,
    style={
        'color': 'white',
        'weight': 1.5,
        'fillOpacity': 0,
        'opacity': 1
    },
    hover_style={
        'color': 'grey',
        'weight': 0,
        'fillOpacity': 0.4
    },
    name='HydroBasins'
)

m.add_layer(polygon_layer)

# info_html = HTML(value='<b>Amazon HydroViewer</b><br>Select variable and time')
# info_control = WidgetControl(widget=info_html, position='topright')
# m.add_control(info_control)

In [6]:
# Add tile server layer to ipyleaflet
import requests

# Configuration
TILE_SERVER_URL = "http://localhost:5000"
variable = "Rainf_tavg"
time_idx = 0
category = 2
profile = 0
vmin = 40  # Minimum value for colormap
vmax = 100  # Maximum value for colormap

# Clear cache first to ensure fresh tiles
try:
    requests.get(f"{TILE_SERVER_URL}/cache/clear")
    print("✓ Cache cleared")
except:
    print("⚠ Could not clear cache")

# Build tile URL template with mode=global and tms=false
# ipyleaflet uses XYZ coordinates (Y=0 at north) by default
# Server will convert XYZ to TMS internally
tile_url = f"{TILE_SERVER_URL}/tiles/{variable}/{time_idx}/{category}/{{z}}/{{x}}/{{y}}.png?colormap=Reds&profile={profile}&mode=global&tms=true&vmin={vmin}&vmax={vmax}"

# Add tile layer (tms=False is default for ipyleaflet - uses XYZ coordinates)
data_layer = TileLayer(
    url=tile_url,
    name=f"{variable} - Category {category}",
    opacity=0.7,
    attribution='HydroViewer'
    # tms=False is the default (XYZ: Y=0 at north)
)

m.add_layer(data_layer)

print(f"✓ Added tile layer: {variable}")
print(f"  vmin={vmin}, vmax={vmax}")
print(f"  mode=global, tms=false (Web Mercator XYZ coordinates)")
print(f"  Transparent NaN values should render correctly in ipyleaflet")

m

✓ Cache cleared
✓ Added tile layer: Rainf_tavg
  vmin=40, vmax=100
  mode=global, tms=false (Web Mercator XYZ coordinates)
  Transparent NaN values should render correctly in ipyleaflet


Map(center=[-7.5, -65.5], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…